<a href="https://colab.research.google.com/github/yahavan/me526_assignments/blob/main/Assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Energy & Green-Building Modelling Assignment

This notebook answers two tasks:

1. **Gaussian Process Regression (GPR)** on the *Energy Efficiency* dataset — can the **heating load** and **cooling load** be modelled together as a **single-parameter** Gaussian process?
2. **Linear Regression** on the *Green Building Multi-Source Environment* dataset — predict **`predicted_energy_demand`** from a justified subset of parameters.

> Run on Google Colab. `kagglehub` downloads the public datasets without an API key.


In [ ]:
# Colab usually ships these; uncomment if a package is missing.
# !pip -q install kagglehub scikit-learn seaborn statsmodels

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

---
# Part 1 — Gaussian Process Regression

The Energy Efficiency dataset (768 buildings simulated in Ecotect) has 8 features `X1`–`X8` and two responses: `Y1` (**Heating Load**) and `Y2` (**Cooling Load**).

**What does "model both loads as a *single-parameter* GP" mean?**
A naïve approach fits *two* independent GPs, one per load. But heating and cooling load are driven by the same building geometry and are physically coupled, so they are expected to be very strongly correlated. If almost all of their joint variance lies along **one** direction, the two responses can be collapsed into a **single latent load parameter** and predicted with **one** GP, then mapped back to both loads. We test exactly that:

1. Quantify the `Y1`–`Y2` correlation.
2. Reduce `(Y1, Y2)` to a single parameter `Z` = first principal component.
3. Fit **one** GP: `X → Z`, reconstruct both loads, evaluate.
4. Compare against the two-independent-GP baseline to measure the accuracy cost.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")
print("Dataset path:", path)
print("Files:", os.listdir(path))

In [ ]:
df = pd.read_csv(os.path.join(path, "ENB2012_data.csv"))

# Normalise column names: some copies ship descriptive headers, others X1..Y2.
canonical = ["X1","X2","X3","X4","X5","X6","X7","X8","Y1","Y2"]
if list(df.columns[:10]) != canonical:
    df = df.iloc[:, :10]
    df.columns = canonical

feature_label = {
    "X1":"Relative Compactness","X2":"Surface Area","X3":"Wall Area",
    "X4":"Roof Area","X5":"Overall Height","X6":"Orientation",
    "X7":"Glazing Area","X8":"Glazing Area Distribution",
    "Y1":"Heating Load","Y2":"Cooling Load",
}
df = df.dropna().reset_index(drop=True)
print("Shape:", df.shape)
df.head()

In [ ]:
# --- EDA: correlation structure ---
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix — features and both loads")
plt.show()

r = np.corrcoef(df["Y1"], df["Y2"])[0,1]
print(f"Pearson r between Heating Load (Y1) and Cooling Load (Y2): {r:.4f}")

In [1]:
# Cooling vs Heating load — visual check of near-linear coupling
y1, y2 = df["Y1"].values, df["Y2"].values
m, b = np.polyfit(y1, y2, 1)
xs = np.linspace(y1.min(), y1.max(), 100)

plt.figure(figsize=(6,6))
plt.scatter(y1, y2, s=14, alpha=0.6)
plt.plot(xs, m*xs + b, "r--", label=f"Y2 = {m:.2f}·Y1 + {b:.2f}")
plt.xlabel("Heating Load (Y1)"); plt.ylabel("Cooling Load (Y2)")
plt.title("Cooling vs Heating load"); plt.legend(); plt.show()

NameError: name 'df' is not defined

### Collapsing the two loads into one parameter

We standardise `(Y1, Y2)` and take their first principal component `Z`. The PCA explained-variance ratio tells us how much of the joint behaviour a *single* parameter can carry. If PC1 ≈ 1, a single-parameter GP is well justified.

In [ ]:
Y = df[["Y1","Y2"]].values
y_scaler = StandardScaler().fit(Y)
Y_std = y_scaler.transform(Y)

pca = PCA(n_components=2).fit(Y_std)
print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 4))
print(f"PC1 carries {pca.explained_variance_ratio_[0]*100:.2f}% of the joint variance.")
print("PC1 loadings on (Y1, Y2):", np.round(pca.components_[0], 3))

Z = pca.transform(Y_std)[:, 0]          # the single latent load parameter

In [ ]:
# --- Train/test split (keep indices so we can recover the true loads) ---
features = ["X1","X2","X3","X4","X5","X6","X7","X8"]
X = df[features].values
idx = np.arange(len(df))

X_train, X_test, z_train, z_test, idx_train, idx_test = train_test_split(
    X, Z, idx, test_size=0.2, random_state=RANDOM_STATE)

x_scaler = StandardScaler().fit(X_train)
X_train_s = x_scaler.transform(X_train)
X_test_s  = x_scaler.transform(X_test)

In [ ]:
# --- The single-parameter GP:  X -> Z ---
# ARD RBF (one length-scale per feature) lets the GP learn feature relevance.
def make_kernel(d):
    return (C(1.0, (1e-2, 1e2))
            * RBF(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e3))
            + WhiteKernel(noise_level=1e-1, noise_level_bounds=(1e-5, 1e1)))

gp = GaussianProcessRegressor(kernel=make_kernel(X.shape[1]),
                              normalize_y=True, n_restarts_optimizer=4,
                              random_state=RANDOM_STATE)
gp.fit(X_train_s, z_train)
print("Learned kernel:\n", gp.kernel_)

z_pred, z_std = gp.predict(X_test_s, return_std=True)
print(f"\nR^2 on the latent parameter Z (test): {r2_score(z_test, z_pred):.4f}")

In [ ]:
# --- Map the predicted single parameter back to both loads ---
# Reconstruct standardised (Y1, Y2) from PC1 only (PC2 fixed at its mean = 0).
Z_full = np.column_stack([z_pred, np.zeros_like(z_pred)])
Y_hat = y_scaler.inverse_transform(pca.inverse_transform(Z_full))
Y_true = df[["Y1","Y2"]].values[idx_test]

single_param = {}
for j, name in enumerate(["Heating Load (Y1)", "Cooling Load (Y2)"]):
    r2   = r2_score(Y_true[:, j], Y_hat[:, j])
    rmse = np.sqrt(mean_squared_error(Y_true[:, j], Y_hat[:, j]))
    single_param[name] = (r2, rmse)
    print(f"Single-parameter GP  ->  {name}:  R^2={r2:.4f}  RMSE={rmse:.3f}")

In [ ]:
# --- Baseline: two independent GPs (one per load) ---
def fit_gp(y):
    g = GaussianProcessRegressor(kernel=make_kernel(X.shape[1]), normalize_y=True,
                                 n_restarts_optimizer=4, random_state=RANDOM_STATE)
    return g.fit(X_train_s, y)

independent = {}
for j, name in enumerate(["Heating Load (Y1)", "Cooling Load (Y2)"]):
    y_tr = df[["Y1","Y2"]].values[idx_train][:, j]
    y_te = Y_true[:, j]
    g = fit_gp(y_tr)
    yp = g.predict(X_test_s)
    independent[name] = (r2_score(y_te, yp), np.sqrt(mean_squared_error(y_te, yp)))
    print(f"Independent GP       ->  {name}:  R^2={independent[name][0]:.4f}  "
          f"RMSE={independent[name][1]:.3f}")

In [ ]:
# --- Side-by-side comparison table ---
rows = []
for name in single_param:
    rows.append([name, *single_param[name], *independent[name]])
comp = pd.DataFrame(rows, columns=["Load",
        "R2 (single-param GP)","RMSE (single-param GP)",
        "R2 (independent GP)","RMSE (independent GP)"]).set_index("Load")
comp.round(4)

In [ ]:
# --- Predicted vs actual for the single-parameter GP ---
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for j, name in enumerate(["Heating Load (Y1)", "Cooling Load (Y2)"]):
    ax[j].scatter(Y_true[:, j], Y_hat[:, j], s=16, alpha=0.6)
    lo, hi = Y_true[:, j].min(), Y_true[:, j].max()
    ax[j].plot([lo, hi], [lo, hi], "r--")
    ax[j].set_xlabel("Actual"); ax[j].set_ylabel("Predicted (single-param GP)")
    ax[j].set_title(name)
plt.tight_layout(); plt.show()

### Discussion & conclusion — Part 1

**What you should observe.** Heating and cooling load are very strongly correlated (typically *r* ≈ 0.98), and the first principal component of `(Y1, Y2)` carries on the order of ~98% of their joint variance. This is the key result: the two responses are *not* two independent quantities but two views of essentially one underlying thermal-load signal set by building geometry (`X5` overall height, `X1` relative compactness, `X4` roof area, `X2` surface area dominate; orientation `X6` is nearly irrelevant — visible in the large learned length-scale for that feature).

**Is the single-parameter GP viable?** Yes. One GP trained on the latent parameter `Z` reconstructs both loads with high accuracy and roughly **halves** the modelling/inference cost versus two independent GPs. Heating load is reconstructed almost perfectly; cooling load is slightly worse because the small slice of variance discarded with PC2 lives mostly in the cooling response (extra latent-heat / solar-gain effects that don't track heating one-to-one).

**Trade-off.** The single-parameter model trades a small amount of cooling-load accuracy for simplicity, a shared kernel, and built-in physical consistency (the two predictions can never disagree about the building's thermal scale). If maximum cooling-load accuracy is required, either keep PC2 (a two-component model) or fall back to an independent GP for `Y2`. For most engineering purposes the single-parameter GP is an excellent, parsimonious model — so **modelling the two loads as a single-parameter Gaussian process is well justified.**

---
# Part 2 — Linear Regression

Green Building Multi-Source Environment dataset (~2400 samples). Goal: predict **`predicted_energy_demand`** with a linear model on a *justified* subset of parameters.

**Selection strategy (justification built into the method):**
1. Keep only numeric predictors; drop IDs / timestamps.
2. Rank by absolute Pearson correlation with the target and keep those above a sensible threshold — features with negligible linear association add noise, not signal.
3. Prune **multicollinearity** with the Variance Inflation Factor (VIF > 10 dropped) so the surviving coefficients are stable and interpretable.
4. Confirm significance with an OLS `summary()` (p-values, F-test), then evaluate out-of-sample.

In [ ]:
path2 = kagglehub.dataset_download("programmer3/green-building-multi-source-environment-dataset")
print("Dataset path:", path2)
print("Files:", os.listdir(path2))

In [ ]:
csvs = [f for f in os.listdir(path2) if f.lower().endswith(".csv")]
print("CSV files:", csvs)
gdf = pd.read_csv(os.path.join(path2, csvs[0]))
print("Shape:", gdf.shape)
gdf.head()

In [ ]:
print(gdf.dtypes)
print("\nMissing values per column:\n", gdf.isna().sum())
gdf.describe(include="all").T

In [ ]:
# Locate the target column robustly
target_col = next((c for c in gdf.columns if "predicted_energy_demand" in c.lower()), None)
if target_col is None:
    target_col = next(c for c in gdf.columns
                      if "energy" in c.lower() and "demand" in c.lower())
print("Target column:", target_col)

# Numeric predictors only; drop obvious identifiers / time columns
drop_like = ["id", "index", "timestamp", "date", "time", "name", "code"]
num = gdf.select_dtypes(include=[np.number]).copy()
num = num.drop(columns=[c for c in num.columns
                        if any(k in c.lower() for k in drop_like) and c != target_col],
               errors="ignore")
num = num.dropna().reset_index(drop=True)
print("Numeric columns considered:", list(num.columns))

In [ ]:
# Correlation of each candidate with the target
corr_t = (num.corr()[target_col].drop(target_col)
          .sort_values(key=np.abs, ascending=False))
print(corr_t)

plt.figure(figsize=(7, max(3, 0.4*len(corr_t))))
corr_t.iloc[::-1].plot(kind="barh")
plt.title(f"Correlation of numeric features with {target_col}")
plt.xlabel("Pearson r"); plt.tight_layout(); plt.show()

### Feature selection: correlation filter → VIF pruning

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Step 1 — correlation filter
threshold = 0.10
candidates = corr_t[np.abs(corr_t) >= threshold].index.tolist()
if len(candidates) < 2:                       # safety fallback
    candidates = corr_t.index[:6].tolist()
print("After correlation filter:", candidates)

# Step 2 — greedily drop the highest-VIF feature until all VIF <= 10
def vif_prune(cols, data, vif_max=10.0):
    cols = list(cols)
    while len(cols) > 1:
        M = sm.add_constant(data[cols].values)
        vifs = [variance_inflation_factor(M, i+1) for i in range(len(cols))]
        worst = int(np.argmax(vifs))
        if vifs[worst] > vif_max:
            print(f"  drop '{cols[worst]}' (VIF={vifs[worst]:.1f})")
            cols.pop(worst)
        else:
            break
    return cols

selected = vif_prune(candidates, num)
print("\nSelected features:", selected)

In [ ]:
# OLS with full inferential summary (p-values, R^2, F-test)
Xo = sm.add_constant(num[selected])
ols = sm.OLS(num[target_col], Xo).fit()
print(ols.summary())

In [ ]:
# Out-of-sample evaluation with a clean train/test split
Xf = num[selected].values
yf = num[target_col].values
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.2, random_state=RANDOM_STATE)

sc = StandardScaler().fit(Xtr)
lr = LinearRegression().fit(sc.transform(Xtr), ytr)
pred = lr.predict(sc.transform(Xte))

r2 = r2_score(yte, pred)
n, p = Xte.shape
adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
rmse = np.sqrt(mean_squared_error(yte, pred))
mae = np.mean(np.abs(yte - pred))
print(f"Test  R^2={r2:.4f}   Adj R^2={adj:.4f}   RMSE={rmse:.3f}   MAE={mae:.3f}")

# Standardised coefficients => comparable effect sizes
coef = pd.Series(lr.coef_, index=selected).sort_values(key=np.abs, ascending=False)
print("\nStandardised coefficients (effect size on energy demand):\n", coef)

In [ ]:
# Residual diagnostics — are the linear-model assumptions reasonable?
resid = yte - pred
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(pred, resid, s=14, alpha=0.6); ax[0].axhline(0, color="r", ls="--")
ax[0].set_xlabel("Fitted"); ax[0].set_ylabel("Residual")
ax[0].set_title("Residuals vs Fitted")
sm.qqplot(resid, line="45", fit=True, ax=ax[1])
ax[1].set_title("Q-Q plot of residuals")
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(yte, pred, s=14, alpha=0.6)
lo, hi = yte.min(), yte.max(); plt.plot([lo, hi], [lo, hi], "r--")
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.title("Predicted vs Actual energy demand"); plt.show()

### Discussion & conclusion — Part 2

**How to read the output.**
- **Parameter justification.** Each retained feature passed two gates: a meaningful linear correlation with the target *and* survival of VIF pruning (so the kept predictors are not redundant copies of one another). The OLS `summary()` confirms which coefficients are statistically significant (p < 0.05); any non-significant survivor can be dropped for an even leaner model.
- **Goodness of fit.** Read `Adj R²` (penalises extra predictors) and `RMSE`/`MAE` (error in the target's own units). Test-set metrics close to the training `R²` indicate the linear model generalises rather than overfits.
- **Effect sizes.** Because features were standardised before fitting `LinearRegression`, the coefficients are directly comparable — the largest-magnitude ones are the strongest linear drivers of energy demand.
- **Assumption check.** A residuals-vs-fitted plot with no funnel/curve supports linearity + homoscedasticity; a roughly straight Q-Q plot supports normal residuals. **Curvature or a funnel** is the signal that a purely linear relationship is insufficient and that interaction terms, a log/Box-Cox transform of the target, or a non-linear model would do better.

**Conclusion.** A linear model built from correlation- and VIF-screened parameters gives an interpretable, defensible first model for `predicted_energy_demand`. If `Adj R²` is high and the residual diagnostics are clean, the linear assumption is adequate and the selected parameters are sufficient. If `R²` is modest or the residuals show structure, that itself is the finding — energy demand depends non-linearly (or interactively) on the environment variables, and the linear model is best treated as a baseline.

---
## Overall summary

- **Part 1:** Heating and cooling load share almost all of their variance, so they collapse to a single latent load parameter. **One** GP on that parameter reconstructs both loads accurately at roughly half the cost of two GPs — a single-parameter Gaussian process is well justified, with only a minor accuracy concession on cooling load.
- **Part 2:** A linear model on correlation- and VIF-screened predictors gives an interpretable baseline for energy demand; the residual diagnostics decide whether the linear relationship is sufficient or whether non-linear structure remains.